# Suite GEN — Sampling and generators

Samples are specified exactly on the random source's 32-bit words, so both implementations draw the same values from the
same seed. `Generate` streams new objects drawn from a distribution: each step chooses a case by weight, and builds what
the case's predicate requires by equality.

In [ ]:
from collections import Counter

from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Patterns import Distributions as D, Generators as G, Predicates as P, Queries as Q, Sampling as M, Validators as V
from mbse.Schemas.Framework import JSON, Stores
from support import Contact, Directory, book, listed, raises


class Words:
    """A random source of the given words, for exact cases."""

    def __init__(self, *words):
        self.words = list(words)

    def next_u32(self):
        return self.words.pop(0)

    def split(self, key):
        return self


APerson = {"person": Contact}
HasName = (P.OfPredicate.Builder().name("HasName").symbols(APerson).parameters(lambda p: p.name("name"))
           .requires(Text.FromFunction(lambda person, name: person.name == name)).create())


def named(weight, name):
    return lambda wt: wt.weight(weight).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, name)))


Names = D.OfWeights.Builder().name("Names").symbols(APerson).decreasing(
    named(10, "alice"), named(5, "ben"), named(15, "chermon"), named(7, "davi")).create()

## GEN-01 · Samples are specified on the source's words

In [ ]:
r = Stores.PCG32(42, 54)
assert [M.uniform(r) for _ in range(3)] == [0.6303102186438938, 0.7270080560068604, 0.7486033647998483]
r = Stores.PCG32(42, 54)
assert [M.below(r, 6) for _ in range(8)] == [3, 3, 2, 1, 1, 4, 5, 3]
r = Stores.PCG32(42, 54)
assert [M.below(r, 2**40) for _ in range(3)] == [788047328265, 208370070163, 325543878766]  # two words each
r = Stores.PCG32(42, 54)
assert [M.weighted(r, [10.0, 5.0, 15.0, 7.0]) for _ in range(10)] == [2, 2, 2, 2, 3, 0, 3, 0, 2, 2]
assert M.uniform(Words(0, 0)) == 0.0 and M.uniform(Words(2**32 - 1, 2**32 - 1)) == 1 - 2**-53  # the ends
assert M.below(Words(2**32 - 1, 5), 3) == 2  # 2**32 - 1 is past the last multiple of 3: rejected, then 5 % 3
assert M.below(Words(7), 1) == 0 and M.weighted(Words(0, 0), [0.0, 2.0, 0.0]) == 1
assert M.weighted(Words(2**32 - 1, 2**32 - 1), [1.0, 2.0, 0.0]) == 1  # the last positive weight, at the top
with raises(ValueError, match="below needs a positive int, got 0"):
    M.below(Words(), 0)
with raises(ValueError, match="a weight must be a finite float, not negative, got -1.0"):
    M.weighted(Words(), [1.0, -1.0])
with raises(ValueError, match="a weight must be a finite float, not negative, got inf"):
    M.weighted(Words(), [1.0, float("inf")])
with raises(ValueError, match="a weight must be a finite float, not negative, got 1"):
    M.weighted(Words(), [1])  # an int: Python only, since TypeScript numbers are floats
with raises(ValueError, match="no weight is positive"):
    M.weighted(Words(), [0.0, 0.0])

## GEN-02 · Generate chooses cases by weight and builds what their equalities require

In [ ]:
store = book()
generated = G.Generate(store, Names, Stores.PCG32(42))
first = [next(generated)["person"].name for _ in range(12)]
assert first == ["chermon", "davi", "chermon", "davi", "alice", "ben", "alice", "davi", "alice", "ben", "chermon", "chermon"]
assert [m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(42)), range(12))] == first  # the same seed
assert [m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(7)), range(12))] != first
assert [m["person"].name for m, _ in zip(G.Generate(book(), Names, Stores.PCG32(42).split("other")), range(12))] != first
names = Counter(next(generated)["person"].name for _ in range(3700))
assert all(abs(names[n] - 100 * w) < 100 for n, w in (("alice", 10), ("ben", 5), ("chermon", 15), ("davi", 7))), names
assert G.settings(Names.cases[0].predicate) == {"person": {"name": "alice"}}
person, age = E.variable("person"), E.variable("person").age
nested = P.OfPredicate.Builder().symbols(APerson).requires(E.literal(30).eq(age).and_(HasName(person, "cy"))).requires(
    person.has("name")).create()
assert G.settings(nested) == {"person": {"age": 30, "name": "cy"}}  # either side of ==, through and and applications
ignored = P.OfPredicate.Builder().symbols(APerson).parameters(lambda p: p.name("n")).requires(
    E.variable("x").name.eq("a")).requires(person.name.eq(E.variable("n"))).requires(person.name.eq(person.age)).requires(
    E.operation("get", person, 1).eq("a")).requires(person.eq("a")).requires(HasName(person, E.variable("n"))).create()
assert G.settings(ignored) == {}  # no symbol's property, or no literal value

## GEN-03 · A case its equalities cannot build is refused

In [ ]:
Grown = D.OfWeights.Builder().symbols(APerson).decreasing(
    lambda wt: wt.weight(1).requires(lambda pred: pred.symbols(APerson).requires(pred.person.age.ge(18)))).create()
with raises(ValueError, match="case 0 cannot be generated from its equalities: what they build does not satisfy it"):
    next(G.Generate(book(), Grown, Stores.PCG32(42)))
Shadowed = D.OfWeights.Builder().symbols(APerson).decreasing(
    lambda wt: wt.weight(1e-9).requires(lambda pred: pred.symbols(APerson).requires(pred.person.has("name"))),
    named(1, "alice")).create()
with raises(ValueError, match="case 1 cannot be generated from its equalities: what they build satisfies case 0"):
    next(G.Generate(book(), Shadowed, Stores.PCG32(42)))
Later = D.OfWeights.Builder().symbols(APerson).decreasing(
    lambda wt: wt.weight(1).requires(lambda pred: pred.symbols(APerson).requires(HasName(pred.person, "alice"))
                                     .requires(pred.person.age.ge(18))),
    named(1e-9, "alice")).create()
with raises(ValueError, match="case 0 cannot be generated from its equalities: what they build does not satisfy it"):
    next(G.Generate(book(), Later, Stores.PCG32(42)))  # a later case holds instead
with raises(ValueError, match="the distribution cannot be drawn from: a distribution needs a case"):
    G.Generate(book(), D.OfWeights.Builder().symbols(APerson).create(), Stores.PCG32(42))


## GEN-04 · Generated data is ordinary data: listed, validated, queried and written

In [ ]:
store = book()
people = [m["person"] for m, _ in zip(G.Generate(store, Names, Stores.PCG32(42)), range(6))]
assert list(store.extent("Contact")) == []  # transient until linked to the store's data
listed(store, *people)
assert len(list(store.extent("Contact"))) == 6
NamedAlice = P.OfPredicate.Builder().name("NamedAlice").symbols({"the": Contact}).requires(HasName(E.variable("the"), "alice")).create()
alices = [m["the"] for m in Q.Scan(store).select(NamedAlice)]
assert alices == [p for p in people if p.name == "alice"] and alices
problems = V.Validate(store, [NamedAlice]).Reachable(Directory, store.singleton("book.Directory"))
assert len(problems) == 6 - len(alices) and all(p.endswith("'NamedAlice' does not hold") for p in problems)
assert '"name": "chermon"' in JSON.ToJSON(store).Reachable(Directory, store.singleton("book.Directory"))